# Assignment 4: Apiori Association 

In [7]:
import pandas as pd

#read csv
df = pd.read_csv("titanic.csv")

print(df.head())
print(df.shape)
print(df.isna().sum())
print(df.columns.to_list())

   Survived  Pclass                                               Name  \
0         0       3                             Mr. Owen Harris Braund   
1         1       1  Mrs. John Bradley (Florence Briggs Thayer) Cum...   
2         1       3                              Miss. Laina Heikkinen   
3         1       1        Mrs. Jacques Heath (Lily May Peel) Futrelle   
4         0       3                            Mr. William Henry Allen   

      Sex   Age  Siblings/Spouses Aboard  Parents/Children Aboard     Fare  
0    male  22.0                        1                        0   7.2500  
1  female  38.0                        1                        0  71.2833  
2  female  26.0                        0                        0   7.9250  
3  female  35.0                        1                        0  53.1000  
4    male  35.0                        0                        0   8.0500  
(887, 8)
Survived                   0
Pclass                     0
Name                      

### Convert numeric features to categorical variables

In [8]:
#remove passenger names
data = df.drop(columns=["Name"]).copy()

#convert survival and passenger class into categories
data["Survived"] = data["Survived"].map({
    0: "Died",
    1: "Survived"
})

data["Pclass"] = data["Pclass"].map({
    1: "First",
    2: "Second",
    3: "Third"
})

#group age into categories
data["Age"] = pd.cut(
    data["Age"],
    bins=[0, 12, 18, 60, float("inf")],
    labels=["Child", "Teen", "Adult", "Senior"],
    include_lowest=True
)

#group fare into categories
data["Fare"] = pd.cut(
    data["Fare"],
    bins=[-1, 10, 30, float("inf")],
    labels=["Low", "Medium", "High"]
)

#group family counts
data["Siblings/Spouses Aboard"] = data["Siblings/Spouses Aboard"].apply(
    lambda x: "None" if x == 0 else "OneOrMore"
)

data["Parents/Children Aboard"] = data["Parents/Children Aboard"].apply(
    lambda x: "None" if x == 0 else "OneOrMore"
)

data.head()

,Survived,Pclass,Sex,Age,Siblings/Spouses Aboard,Parents/Children Aboard,Fare
0,Died,Third,male,Adult,OneOrMore,None,Low
1,Survived,First,female,Adult,OneOrMore,None,High
2,Survived,Third,female,Adult,None,None,Low
3,Survived,First,female,Adult,OneOrMore,None,High
4,Died,Third,male,Adult,None,None,Low


#### One hot tencoding the data 

In [9]:
transactions = pd.get_dummies(
    data.astype(str),
    prefix_sep="=",
    dtype=bool
)

print(transactions.shape)
transactions.head()

(887, 18)


,Survived=Died,Survived=Survived,Pclass=First,Pclass=Second,Pclass=Third,Sex=female,Sex=male,Age=Adult,Age=Child,Age=Senior,Age=Teen,Siblings/Spouses Aboard=None,Siblings/Spouses Aboard=OneOrMore,Parents/Children Aboard=None,Parents/Children Aboard=OneOrMore,Fare=High,Fare=Low,Fare=Medium
0,True,False,False,False,True,False,True,True,False,False,False,False,True,True,False,False,True,False
1,False,True,True,False,False,True,False,True,False,False,False,False,True,True,False,True,False,False
2,False,True,False,False,True,True,False,True,False,False,False,True,False,True,False,False,True,False
3,False,True,True,False,False,True,False,True,False,False,False,False,True,True,False,True,False,False
4,True,False,False,False,True,False,True,True,False,False,False,True,False,True,False,False,True,False


## Part A Run Apiori 

In [12]:
from mlxtend.frequent_patterns import apriori, association_rules

#find frequent itemsets with minimum support of 20%
frequent_itemsets = apriori(
    transactions, 
    min_support=.20, 
    use_colnames=True
)

#generate association rules with minimum confidence of 70%
rules = association_rules(
    frequent_itemsets, 
    metric="confidence",
    min_threshold=.70
)

#display important metrics
rules[[
    "support",
    "confidence",
    "lift"
]].head(20)

,support,confidence,lift
0,0.414882,0.755647,1.229833
1,0.523112,0.851376,1.317924
2,0.523112,0.809773,1.317924
3,0.491545,0.800000,1.021007
4,0.444194,0.722936,1.061662
5,0.497182,0.809174,1.064893
6,0.299887,0.798799,1.300063
7,0.262683,0.742038,1.924526
8,0.291995,0.757310,0.966524
9,0.208568,0.856481,1.093092


### Running 3 Different Experiments

In [14]:
#support and confidence
experiments = [
    (0.20, 0.70),
    (0.15, 0.60),
    (0.10, 0.50)
]

all_rules = {}

for support, confidence in experiments:
    frequent_itemsets = apriori(
        transactions,
        min_support=support,
        use_colnames=True
    )

    rules = association_rules(
        frequent_itemsets,
        metric="confidence",
        min_threshold=confidence
    )

    all_rules[(support, confidence)] = rules

    print(f"\nSupport: {support}, Confidence: {confidence}")
    print(f"Frequent itemsets: {len(frequent_itemsets)}")
    print(f"Association rules: {len(rules)}")

    display(rules[[
        "support",
        "confidence",
        "lift"
    ]].head(10))


Support: 0.2, Confidence: 0.7
Frequent itemsets: 151
Association rules: 944


,support,confidence,lift
0,0.414882,0.755647,1.229833
1,0.523112,0.851376,1.317924
2,0.523112,0.809773,1.317924
3,0.491545,0.800000,1.021007
4,0.444194,0.722936,1.061662
5,0.497182,0.809174,1.064893
6,0.299887,0.798799,1.300063
7,0.262683,0.742038,1.924526
8,0.291995,0.757310,0.966524
9,0.208568,0.856481,1.093092



Support: 0.15, Confidence: 0.6
Frequent itemsets: 185
Association rules: 1283


,support,confidence,lift
0,0.414882,0.755647,1.229833
1,0.414882,0.675229,1.229833
2,0.523112,0.851376,1.317924
3,0.523112,0.809773,1.317924
4,0.491545,0.627338,1.021007
5,0.491545,0.800000,1.021007
6,0.444194,0.722936,1.061662
7,0.444194,0.652318,1.061662
8,0.497182,0.809174,1.064893
9,0.497182,0.654303,1.064893



Support: 0.1, Confidence: 0.5
Frequent itemsets: 271
Association rules: 1843


,support,confidence,lift
0,0.109357,0.527174,0.857988
1,0.414882,0.755647,1.229833
2,0.414882,0.675229,1.229833
3,0.523112,0.851376,1.317924
4,0.523112,0.809773,1.317924
5,0.491545,0.627338,1.021007
6,0.491545,0.800000,1.021007
7,0.444194,0.722936,1.061662
8,0.444194,0.652318,1.061662
9,0.170237,0.533569,0.868396


## Part B Rank top 5 by confidence

In [15]:

#using rules from experiment 3 since it has the lowest threshold
rules = all_rules[(0.10, 0.50)]

top_confidence = rules.sort_values(
    by=["confidence", "support"],
    ascending=False
).head(5)

display(top_confidence[[
    "support",
    "confidence",
    "lift"
]])

,support,confidence,lift
855,0.264938,0.995763,1.310447
1209,0.254791,0.995595,1.310226
1368,0.253664,0.995575,1.310201
1295,0.251409,0.995536,1.310149
1544,0.241263,0.995349,1.309903


## Part c: Rank the top 5 by lift

In [17]:
top_lift = rules.sort_values(
    by=["lift", "support"],
    ascending=False
).head(5)

display(top_lift[[
    "support",
    "confidence",
    "lift"
]])

,support,confidence,lift
676,0.120631,0.578378,3.857305
678,0.120631,0.804511,3.857305
635,0.111612,0.562500,3.668658
636,0.111612,0.727941,3.668658
631,0.111612,0.891892,3.662538


In [21]:
#format association rules into readable text
def format_rule(row):
    left = ", ".join(sorted(row["antecedents"]))
    right = ", ".join(sorted(row["consequents"]))
    return f"{left} → {right}"

rules = all_rules[(0.10, 0.50)].copy()

rules["Rule"] = rules.apply(format_rule, axis=1)

#ranking top 5 by confidence
top_confidence = rules.sort_values(
    by=["confidence", "support"],
    ascending=False
).head(5)

pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", 200)

print("Top 5 Rules by Confidence")
display(top_confidence[[
    "Rule", "support", "confidence", "lift"
]])

#ranking top 5 by lift
top_lift = rules.sort_values(
    by=["lift", "support"],
    ascending=False
).head(5)

pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", 200)

print("Top 5 Rules by Lift")
display(top_lift[[
    "Rule", "support", "confidence", "lift"
]])

Top 5 Rules by Confidence


,Rule,support,confidence,lift
855,"Age=Adult, Fare=Low, Sex=male → Parents/Children Aboard=None",0.264938,0.995763,1.310447
1209,"Fare=Low, Sex=male, Siblings/Spouses Aboard=None, Survived=Died → Parents/Children Aboard=None",0.254791,0.995595,1.310226
1368,"Age=Adult, Fare=Low, Sex=male, Siblings/Spouses Aboard=None → Parents/Children Aboard=None",0.253664,0.995575,1.310201
1295,"Age=Adult, Fare=Low, Pclass=Third, Sex=male → Parents/Children Aboard=None",0.251409,0.995536,1.310149
1544,"Fare=Low, Pclass=Third, Sex=male, Siblings/Spouses Aboard=None, Survived=Died → Parents/Children Aboard=None",0.241263,0.995349,1.309903


Top 5 Rules by Lift


,Rule,support,confidence,lift
676,"Age=Adult, Pclass=First → Fare=High, Parents/Children Aboard=None",0.120631,0.578378,3.857305
678,"Fare=High, Parents/Children Aboard=None → Age=Adult, Pclass=First",0.120631,0.804511,3.857305
635,"Age=Adult, Fare=High → Pclass=First, Survived=Survived",0.111612,0.562500,3.668658
636,"Pclass=First, Survived=Survived → Age=Adult, Fare=High",0.111612,0.727941,3.668658
631,"Age=Adult, Fare=High, Survived=Survived → Pclass=First",0.111612,0.891892,3.662538
